In [1]:
import numpy as np
import pandas as pd

Minkowski distance $$||X||_p=(\sum_{i=1}^{m}|x_i|^{p})^{1/p}$$ \
In case of a tie the algorithm calculates sum of distances from k subset for tied classes and pickes the one with smaller sum

In [2]:
class KnnClassifier:
    def __init__(self,k,p=2):
        self.k=k
        self.p=p
    def fit(self,X,y):
        self.X=np.asarray(X)
        self.y=np.asarray(y)
    def predict(self,X):
        X = np.asarray(X)
        preds=[]
        for sample in X:
            dist=np.linalg.norm(self.X-sample,ord=self.p,axis=1)
            idx=np.argsort(dist)[:self.k]
            y_k=self.y[idx]
            unique_c,counts=np.unique(y_k,return_counts=True)
            max_count=np.max(counts)
            tied=unique_c[counts==max_count]
            if len(tied)>1:
                tied_dist=[]
                for c in tied:
                    tied_dist.append(np.sum(dist[idx[y_k==c]]))
                preds.append(tied[np.argmin(tied_dist)])
            else:
                preds.append(tied[0])
        return preds

In [3]:
df=pd.read_csv('data/iris.csv')

In [4]:
X,y=df.loc[:,['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']],df.loc[:,'Species']

In [5]:
y=y.map({'Iris-setosa':0,'Iris-versicolor':1,'Iris-virginica':2})

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)

In [7]:
X_train_std=X_train.std(axis=0)
X_train_mean=X_train.mean(axis=0)
X_train=(X_train-X_train_mean)/X_train_std
X_test=(X_test-X_train_mean)/X_train_std

In [9]:
knn=KnnClassifier(5)
knn.fit(X_train.values,y_train.values)
preds=knn.predict(X_test.values)
print(f"Accuracy: {np.mean(preds==y_test)}%")

Accuracy: 0.9333333333333333%
